# Baseline: fine-tuned voice + content network (follows the project brief)

Implements the architecture of `claude_voice_content_finetuning_brief.md` end to end, on one GPU (RTX 5090 or Colab) or, slowly, MPS/CPU:

```
segmented audio + text per speaker turn
   -> audio encoder (WavLM, top layers FINE-TUNED, attentive pooling, learnable layer weights)
   -> text encoder  (DeBERTa-v3, top layers FINE-TUNED or LoRA)
   -> multimodal fusion (audio + text + acoustic scalars) with MODALITY DROPOUT
   -> SIGNAL heads only: emotion, arousal, delivery                                   [Stage A = signal model]
   -> measured per-turn signals: pitch level/spread (semitones), voiced fraction, loudness, pause fraction,
      speech rate, filler rate, words; each also as a delta vs the speaker's own earlier turns (tone drift, ...)
      -> signal_timeline (PRIMARY output) + response latency from conversation events
   -> causal GRU over each speaker's turns in a call (signals + deltas + speech embedding)  [Stage B]
   -> optional bluff / fabrication / mismatch model scores (SECONDARY) -> live hints + post-call JSON
```

**Signals first (decision 2026-10-10, Dae; see `docs/finetuning_plan.md` decision update).** Product: a credit-score-like trust profile for voice interactions with stakes. Demo: a scam-call simulation where a **human plays the scammer** and our AI applies pressure (e.g. the bank agent asking verification questions); the Digital Twin dashboard shows the scammer's signals (tone drift, speech rate, hesitation, de-escalation/response latency, emotion, arousal, delivery, each vs their own baseline) and decisions, backed by a verifiable credential. Stage B's deception readings are optional and secondary: this run has only 6 bluff and 12 mismatch labels (they report `insufficient_evidence`); fabrication has 180 synthetic audio labels plus Diplomacy text. An LLM could read `signal_timeline` instead (not built). **5090 run:** `TIME_BUDGET_MIN = 90` sizes Stage A from measured speed; see `docs/train_on_gpu_machine.md`.

What is real and what is not (read this before trusting any number):
- **Fine-tuned, not frozen:** the top `UNFREEZE_AUDIO_TOP` WavLM layers and top `UNFREEZE_TEXT_TOP` DeBERTa layers train with the heads. Lower layers stay frozen (small labelled sets). No foundation model is trained from scratch.
- **Targets and data.** Stage A trains: emotion (acted + TV dialogue), arousal, **delivery mode (read vs spontaneous)**. The **deception-style targets (negotiating bluff, fabrication, interest mismatch) are optional readings scored only by Stage B**, from the signals and the speech; one possible interpretation, never a verdict. Open ungated labelled data is **text-only** (Diplomacy: the sender's own report of an intended lie, wired as `fabrication`; no ungated open deception set with audio was found, see `docs/research/deception_delivery_datasets.md`). Scripted ElevenLabs calls (`scripts/generate_synthetic_calls.py`) add audio with labels by construction (bluff against a documented private floor, mismatch against documented private intent, scripted false claims, read vs spontaneous delivery), but they are all TTS, so they validate the pipeline and teach TTS-style delivery, not real speech. Audio's contribution to deception on real speech is untested; heads with no labels stay untrained and the output contract returns `null`, as the brief requires.
- **Delivery mode is corpus-confounded** (read = LibriSpeech, spontaneous = AMI): the shortcut audit cell shows how far scalar channel features alone get. Treat it as a pipeline test, not a validated read/rehearsed detector.
- All scores are **model scores**; `calibration_status` is always `not_established`. Emotion models trained on acted speech are not stress or intent detectors, and nothing here validates hidden-intention detection. Consent-based use only.

In [ ]:
# ---- Config -----------------------------------------------------------------------------------
QUICK = False                         # True: tiny smoke run (minutes, any machine). False: full run (use the GPU).
AUDIO_CKPT = "microsoft/wavlm-base-plus"   # alt (research-only licence): "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"
TEXT_CKPT = "microsoft/deberta-v3-base"
UNFREEZE_AUDIO_TOP = 1 if QUICK else 4      # top transformer layers of the audio encoder that train
UNFREEZE_TEXT_TOP = 1 if QUICK else 2
USE_LORA_TEXT = False                       # needs `pip install peft`; LoRA on DeBERTa attention instead of unfreezing layers
DATASETS = ["ravdess", "meld", "ami", "librispeech", "deception", "synthetic"]   # deception = Diplomacy (text only); synthetic = ElevenLabs calls with scripted labels
MELD_SPLITS = ["dev"] if QUICK else ["train", "dev", "test"]
FULL_DATA = True                      # full run: AMI spread across train/validation/test (see AMI_MAX_SHARDS) and LibriSpeech test-clean + dev-clean + train-clean-100. False: the small subset used for smoke tests
AMI_MAX_SHARDS = 10                   # AMI is ~134k segments in 51 shards (15 GB): a spread of 10 evenly spaced shards (~27k segments, ~3 GB) is plenty at equal dataset weight. None = all 51
AMI_SHARDS = ["train-00000-of-00042"] if QUICK else (None if FULL_DATA else ["train-00000-of-00042", "train-00001-of-00042", "validation-00000-of-00005"])   # None = list every ihm shard
LIBRI_SETS = ["test-clean"] if (QUICK or not FULL_DATA) else ["test-clean", "dev-clean"]   # train-clean-100 dropped: 6.3 GB from openslr (~3 MB/s) costs ~30 min of the time budget
PER_DS_PER_EPOCH = 40 if QUICK else 8000   # rows drawn from EACH dataset per epoch (equal weight per dataset). Big sets are walked through without replacement across epochs, so every row is used; small sets recycle
CACHE_GB = 8.0                                                # in-RAM audio cache limit (float16); least recently used entries are dropped
QUICK_N = int(__import__("os").environ.get("QUICK_N", 240))   # rows per dataset in QUICK mode
SEED = 0
D = 256
EPOCHS_A = 1 if QUICK else 12            # 12 x 8000 = 96k draws per dataset: every loaded AMI / LibriSpeech row is seen at least once
EPOCHS_B = 4 if QUICK else 30
BATCH = 8 if QUICK else 32            # 32 GB on the 5090: larger batches keep the GPU busy (was 16)
EVAL_BATCH = 32 if QUICK else 96
MAX_AUDIO_S = 6.0
MAX_TOK = 64
LR_ENC, LR_HEAD = (2e-5, 1e-3) if QUICK else (2.8e-5, 1.4e-3)   # sqrt-scaled for batch 32; warm-up + cosine decay over the budgeted steps (Stage A cell)
P_MOD_DROP = 0.3
TIME_BUDGET_MIN = None if QUICK else 90     # whole-notebook wall-clock limit (from running this cell); Stage A measures its speed and shrinks PER_DS_PER_EPOCH / EPOCHS_A / AUDIT_MAX to fit. None = no limit
AUDIT_MAX = 30_000                          # train rows used by the scalar-only shortcut audit (may be lowered by the time budget)
VAL_MAX = 2_000                             # val rows scored after each Stage A epoch to pick the best epoch (random subsample if larger)
STAGE_A_TASKS = ("emotion", "arousal", "delivery")   # Stage A = signal model. bluff / fabrication / mismatch are scored only by Stage B, from these signals + measured prosody + the speech embedding
N_MIN_LABELS = 200                          # a target head only emits scores if it saw at least this many labels

import os, sys, json, time, random, hashlib, zipfile, tarfile, urllib.request, io
T_START = time.time()
N_LOAD_THREADS = min(16, os.cpu_count() or 4)   # parallel audio decode/resample threads (librosa/soundfile/soxr release the GIL)
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
def seed_all(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(SEED)
DEVICE_OVERRIDE = os.environ.get("DEVICE_OVERRIDE")   # set to "cpu" if MPS crashes (DeBERTa hits an Apple-GPU matmul assertion); CUDA is the intended target
DEVICE = DEVICE_OVERRIDE or ("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
REPO = Path("..") if (Path("..") / "backend").exists() else None
WORK = (REPO / "data" / "external") if REPO else Path("work")
OUT = (REPO / "results") if REPO else Path("results")
MODELS = (REPO / "data" / "models") if REPO else Path("models")
for p in (WORK, OUT, MODELS): p.mkdir(parents=True, exist_ok=True)
if DEVICE == "cuda":   # Blackwell (RTX 5090): TF32 for any fp32 matmul/conv outside bf16 autocast
    torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True; torch.set_float32_matmul_precision("high")
print("device:", DEVICE, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
try:
    import pandas, pyarrow, transformers, librosa, soundfile, sklearn, sentencepiece, google.protobuf, huggingface_hub  # noqa: F401
except ImportError:
    %pip install -q pandas pyarrow transformers librosa soundfile scikit-learn sentencepiece protobuf huggingface_hub tqdm

## 1. Data schema and adapters
One row per speaker **segment** (turn): `dataset, call_id, speaker, order (time or turn index), t_start, t_end, path, text, group` plus label columns. **Unknown labels stay unknown** (`None`/`-1`, masked out of every loss; never turned into negatives). Splits are **by `group`** (actor, TV episode, meeting, speaker), so no group is in two splits.

| Dataset | Gives | Labels | Group |
|---|---|---|---|
| RAVDESS (acted, CC BY-NC-SA) | audio, 2 fixed sentences | emotion, arousal | actor |
| MELD (TV dialogue, calls = dialogues) | audio, text, speaker, order | emotion, arousal | episode |
| AMI (meetings, CC-BY) | audio, text, speaker, times | delivery = spontaneous | meeting |
| LibriSpeech (test-clean; full run adds dev-clean and train-clean-100) (CC BY) | audio, text | delivery = read | speaker |
| Diplomacy (HF, text only) | text, speaker, order; **no audio** | `fabrication` = sender's own report of an intended lie (not independently verified) | game |
| Synthetic calls (ElevenLabs, `scripts/generate_synthetic_calls.py`) | audio, text, speakers, times | scripted: delivery, intended emotion, fabrication, bluff (documented private floor), mismatch (documented private intent) | call |

In [ ]:
EMO = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
HIGH = {"happy", "angry", "fearful", "surprised"}
TASKS = {"emotion": EMO, "arousal": ["low", "high"], "delivery": ["read", "rehearsed", "spontaneous"],
         "bluff": ["no", "yes"], "fabrication": ["no", "yes"], "mismatch": ["no", "yes"]}
LABEL_COLS = list(TASKS)
SR = 16_000

def md5(p, n=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(n), b""): h.update(b)
    return h.hexdigest()

def row(**k):
    r = dict(dataset="", call_id="", speaker="", order=0.0, t_start=np.nan, t_end=np.nan, path="", text="", group="")
    r.update({c: None for c in LABEL_COLS}); r.update(k); return r

def load_ravdess():
    root = WORK / "ravdess" / "speech"
    if not any(root.rglob("*.wav")):
        root.mkdir(parents=True, exist_ok=True); z = WORK / "ravdess" / "Audio_Speech_Actors_01-24.zip"
        if not z.exists():
            print("downloading RAVDESS ..."); urllib.request.urlretrieve("https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1", z)
        assert md5(z) == "bc696df654c87fed845eb13823edef8a", "RAVDESS md5 mismatch"
        zipfile.ZipFile(z).extractall(root)
    st = {1: "Kids are talking by the door", 2: "Dogs are sitting by the door"}; rows = []
    for i, p in enumerate(sorted(root.rglob("*.wav"))):
        f = [int(x) for x in p.stem.split("-")]; e = EMO[f[2] - 1]
        rows.append(row(dataset="ravdess", call_id=f"rav_{f[6]:02d}", speaker=f"rav_{f[6]:02d}", order=i, path=str(p), text=st[f[4]],
                        group=f"rav_{f[6]:02d}", emotion=e, arousal="high" if e in HIGH else "low"))
    return rows

MELD_MAP = {"anger": "angry", "disgust": "disgust", "fear": "fearful", "joy": "happy", "neutral": "neutral", "sadness": "sad", "surprise": "surprised"}
def load_meld():
    # HF repo ajyy/MELD_audio: <split>.csv + archive/<split>.tar.gz with <split>/dia{D}_utt{U}.flac (repo's loading script bypassed)
    from huggingface_hub import hf_hub_download
    rows = []
    for sp in MELD_SPLITS:
        meta = pd.read_csv(hf_hub_download("ajyy/MELD_audio", f"{sp}.csv", repo_type="dataset")); dest = WORK / "meld"; (dest / sp).mkdir(parents=True, exist_ok=True)
        if not any((dest / sp).glob("*.flac")):
            with tarfile.open(hf_hub_download("ajyy/MELD_audio", f"archive/{sp}.tar.gz", repo_type="dataset")) as t: t.extractall(dest)
        for _, r in meta.iterrows():
            p = dest / sp / f"dia{r['Dialogue_ID']}_utt{r['Utterance_ID']}.flac"; e = MELD_MAP.get(str(r["Emotion"]).lower())
            if p.exists() and e:
                rows.append(row(dataset="meld", call_id=f"meld_{sp}_{r['Dialogue_ID']}", speaker=f"meld_{r['Speaker']}", order=float(r["Utterance_ID"]), path=str(p),
                                text=str(r["Utterance"]), group=f"meld_s{r['Season']}e{r['Episode']}", emotion=e, arousal="high" if e in HIGH else "low"))
    return rows

def load_ami():
    # edinburghcstr/ami ihm parquet shards (CC-BY-4.0): audio bytes, meeting_id, speaker_id, begin_time, end_time, text
    import pyarrow.parquet as pq, soundfile as sf
    from huggingface_hub import HfApi, hf_hub_download
    wavdir = WORK / "ami_wav"; wavdir.mkdir(parents=True, exist_ok=True); rows = []
    shards = AMI_SHARDS or sorted(f[:-len(".parquet")] for f in HfApi().list_repo_files("edinburghcstr/ami", repo_type="dataset") if f.startswith("ihm/") and f.endswith(".parquet"))
    shards = [s.removeprefix("ihm/") for s in shards]
    if AMI_MAX_SHARDS and len(shards) > AMI_MAX_SHARDS: shards = [shards[int(k * len(shards) / AMI_MAX_SHARDS)] for k in range(AMI_MAX_SHARDS)]   # evenly spaced: spans test/train/validation meetings
    print("AMI shards:", len(shards))
    for sh in shards:
        pf = pq.ParquetFile(hf_hub_download("edinburghcstr/ami", f"ihm/{sh}.parquet", repo_type="dataset"))
        for b in pf.iter_batches(batch_size=64, columns=["audio", "meeting_id", "speaker_id", "begin_time", "end_time", "text"]):
            for r in b.to_pylist():
                if r["end_time"] - r["begin_time"] < 1.0 or not r["text"].strip(): continue
                p = wavdir / f"{r['meeting_id']}_{r['speaker_id']}_{int(r['begin_time'] * 100):07d}.wav"
                if not p.exists():
                    a, sr = sf.read(io.BytesIO(r["audio"]["bytes"]), dtype="float32"); sf.write(p, a if a.ndim == 1 else a.mean(1), sr)
                rows.append(row(dataset="ami", call_id=r["meeting_id"], speaker=r["speaker_id"], order=r["begin_time"], t_start=r["begin_time"], t_end=r["end_time"],
                                path=str(p), text=r["text"].lower(), group=r["meeting_id"], delivery="spontaneous"))
    return rows

def load_librispeech():
    rows = []
    for name in LIBRI_SETS:
        root = WORK / "LibriSpeech" / name
        if not root.exists():
            tgz = WORK / f"{name}.tar.gz"
            if not tgz.exists(): print(f"downloading LibriSpeech {name} ..."); urllib.request.urlretrieve(f"https://www.openslr.org/resources/12/{name}.tar.gz", tgz)
            with tarfile.open(tgz) as t: t.extractall(WORK)
        for tr in sorted(root.rglob("*.trans.txt")):
            spk = tr.parent.parent.name
            for i, line in enumerate(tr.read_text().splitlines()):
                uid, txt = line.split(" ", 1); p = tr.parent / f"{uid}.flac"
                rows.append(row(dataset="librispeech", call_id=f"{spk}-{tr.parent.name}", speaker=spk, order=i, path=str(p), text=txt.lower(), group=f"ls_{spk}", delivery="read"))
    return rows

def load_deception():
    # Diplomacy (HF community-datasets/diplomacy_detection): TEXT-ONLY game dialogue; sender_labels = the sender's OWN report of whether
    # the message was truthful (1 = truthful, 0 = intended lie, ~4.5%). It is intent reported in a game, not independently verified
    # deception and not commercial. Mapped to `fabrication` (knowingly false statement). No audio: the audio and scalar branches are masked.
    from huggingface_hub import HfApi, hf_hub_download
    rows = []
    for f in HfApi().list_repo_files("community-datasets/diplomacy_detection", repo_type="dataset"):
        if not f.endswith(".parquet"): continue
        d = pd.read_parquet(hf_hub_download("community-datasets/diplomacy_detection", f, repo_type="dataset"))
        for ri, r in d.iterrows():
            for m, sp, lab, oi in zip(r["messages"], r["speakers"], r["sender_labels"], r["relative_message_index"]):
                rows.append(row(dataset="diplomacy", call_id=f"dip_{r['game_id']}_{Path(f).stem}_{ri}", speaker=f"p{sp}", order=float(oi), path="", text=str(m),
                                group=f"dip_{r['game_id']}", fabrication="no" if int(lab) == 1 else "yes"))
    return rows


def load_synthetic():
    # ElevenLabs-generated calls with labels BY CONSTRUCTION (scripts/generate_synthetic_calls.py -> data/synthetic/<run>/calls.jsonl).
    # Pass the run folder via SYNTH_RUN or the newest folder is used. Labels: delivery (read/spontaneous), emotion (customer, intended),
    # arousal (derived), fabrication (scripted false claims), bluff (documented private floor), mismatch (documented private intent).
    base = (REPO / "data" / "synthetic") if REPO else Path("synthetic")
    runs = sorted(p for p in base.glob("*") if (p / "calls.jsonl").exists())
    if not runs: print("no synthetic run found; run scripts/generate_synthetic_calls.py first"); return []
    run = Path(os.environ["SYNTH_RUN"]) if "SYNTH_RUN" in os.environ else runs[-1]; rows = []
    for line in (run / "calls.jsonl").read_text().splitlines():
        c = json.loads(line)
        for k, s in enumerate(c["segments"]):
            a = s["annotations"]; wav = run / s["audio_file"]
            if not wav.exists(): continue
            e = a.get("intended_emotion")
            rows.append(row(dataset="synthetic", call_id=c["call_id"], speaker=s["speaker_id"], order=float(k), t_start=s["start_seconds"], t_end=s["end_seconds"], path=str(wav),
                            text=s["text"], group=c["call_id"], emotion=e if e in EMO else None, arousal=("high" if e in HIGH else "low") if e in EMO else None,
                            delivery=a.get("delivery_mode"), fabrication=a.get("fabrication"), bluff=a.get("negotiating_bluff"), mismatch=a.get("mismatch")))
    return rows

ADAPTERS = {"ravdess": load_ravdess, "meld": load_meld, "ami": load_ami, "librispeech": load_librispeech, "deception": load_deception, "synthetic": load_synthetic}
frames = []
for d in DATASETS:
    r = pd.DataFrame(ADAPTERS[d]())
    if QUICK and len(r) > QUICK_N: r = r.sort_values(["call_id", "order"]).iloc[:QUICK_N]
    frames.append(r)
df = pd.concat(frames, ignore_index=True).reset_index(drop=True)
print(len(df), "segments,", df.call_id.nunique(), "calls,", df.group.nunique(), "groups")
print(df.groupby("dataset")[LABEL_COLS].agg(lambda s: s.notna().sum()))

### Splits (group-disjoint) and label encoding
70/15/15 per dataset by group, seeded. Labels become integers, `-1` = unknown (ignored by every loss).

In [ ]:
rng = np.random.default_rng(SEED); split = {}
for d, sub in df.groupby("dataset"):
    gs = sorted(sub.group.unique()); rng.shuffle(gs); n = len(gs)
    nt, nv = max(1, round(0.15 * n)), max(1, round(0.15 * n)) if n >= 3 else 0
    for i, g in enumerate(gs): split[g] = "test" if i < nt else ("val" if i < nt + nv else "train")
df["split"] = df.group.map(split)
Y = np.full((len(df), len(TASKS)), -1, dtype=np.int64)
for j, (t, classes) in enumerate(TASKS.items()):
    m = {c: i for i, c in enumerate(classes)}
    Y[:, j] = df[t].map(lambda v: m.get(v, -1) if v is not None and v == v else -1).values
print(df.split.value_counts().to_dict(), "| labelled per task:", {t: int((Y[:, j] >= 0).sum()) for j, t in enumerate(TASKS)})
assert not set(df[df.split == "train"].group) & set(df[df.split == "test"].group)

## 2. Stage A: segment network (audio + text + scalars, fine-tuned top layers, modality dropout)
Audio branch: encoder hidden states -> learnable softmax weights over layers -> **attentive pooling** over frames. Text branch: mean-pooled encoder states. Scalar branch: loudness (NOT normalised away), peak, clipping, pause fraction, zero-crossing rate, duration. Fusion: projected embeddings + modality-present flags. **Modality dropout** trains audio-only and text-only predictions inside the same model; the ablation below masks one modality without touching the other.

In [ ]:
import librosa, re
import soundfile as sf
from transformers import AutoModel, AutoTokenizer, WavLMModel, AutoModelForAudioClassification

def scalar_feats(y):
    fr = librosa.util.frame(y, frame_length=400, hop_length=160) if len(y) >= 400 else np.zeros((400, 1), dtype=np.float32)
    db = 20 * np.log10(np.sqrt((fr ** 2).mean(0) + 1e-12) + 1e-6)
    act = db > max(np.percentile(db, 10) + 10, np.percentile(db, 95) - 25)
    return np.array([len(y) / SR, db[act].mean() if act.any() else db.mean(), np.abs(y).max(), np.mean(np.abs(y) >= 0.99),
                     1 - act.mean(), np.mean(np.abs(np.diff(np.signbit(y).astype(np.int8)))) if len(y) > 1 else 0, np.percentile(db, 95) - np.percentile(db, 10)], dtype=np.float32)
N_SCALAR = 7

# ---- Measured per-turn signals (interpretable inputs to Stage B; deltas vs the speaker's own earlier turns give tone drift etc.) ----
FILLERS = re.compile(r"\b(um+|uh+|erm|er|ah|hmm+|like|you know|i mean|sort of|kind of|i guess|maybe|probably|i think)\b", re.I)
SIG_NAMES = ["pitch_st_median", "pitch_st_spread", "voiced_frac", "loudness_db", "pause_frac", "speech_rate_wps", "filler_rate", "log_words", "has_audio"]
def prosody(y):
    """pitch level and variation in semitones re 100 Hz over voiced (energetic) frames, plus voiced fraction"""
    if len(y) < 2048: return np.zeros(3, np.float32)
    f0 = librosa.yin(y, fmin=65, fmax=400, sr=SR, frame_length=1024, hop_length=320)
    rms = librosa.feature.rms(y=y, frame_length=1024, hop_length=320)[0]; n = min(len(f0), len(rms)); f0, rms = f0[:n], rms[:n]
    v = rms > max(np.percentile(rms, 30), 1e-4)
    if v.sum() < 5: return np.array([0.0, 0.0, float(v.mean())], np.float32)
    st = 12 * np.log2(f0[v] / 100.0); q25, q50, q75 = np.percentile(st, [25, 50, 75])
    return np.array([q50, (q75 - q25) / 1.35, v.mean()], np.float32)                           # robust spread: yin octave errors inflate std
def turn_signals(i):
    txt = df.text.iat[i] or ""; nw = len(txt.split()); fill = len(FILLERS.findall(txt)) / max(nw, 1)
    if df.path.iat[i] == "": return np.array([0, 0, 0, 0, 0, 0, fill, np.log1p(nw), 0], np.float32)
    y = get_audio(i, False); sc = scalar_feats(y); pr = prosody(y); ts, te = df.t_start.iat[i], df.t_end.iat[i]
    dur = (te - ts) if (ts == ts and te == te and te > ts) else sf.info(df.path.iat[i]).duration
    rate = min(nw / dur, 8.0) if dur > 0.3 and nw else 0.0
    return np.array([pr[0], pr[1], pr[2], sc[1], sc[4], rate, fill, np.log1p(nw), 1], np.float32)

from collections import OrderedDict, deque
import threading
from concurrent.futures import ThreadPoolExecutor
_audio_cache = OrderedDict(); _cache_bytes = [0]; _cache_lock = threading.Lock(); _pool = ThreadPoolExecutor(N_LOAD_THREADS)
def get_audio(i, train):
    p = df.path.iat[i]
    if p == "": return np.zeros(1600, dtype=np.float32)                                      # text-only row: audio branch is masked
    with _cache_lock:
        y16 = _audio_cache.get(p)
        if y16 is not None: _audio_cache.move_to_end(p)
    if y16 is None:
        y, _ = librosa.load(p, sr=SR)
        if df.dataset.iat[i] in ("ravdess",): y, _ = librosa.effects.trim(y, top_db=30)
        y16 = y.astype(np.float16)
        with _cache_lock:
            if p not in _audio_cache: _audio_cache[p] = y16; _cache_bytes[0] += y16.nbytes
            while _cache_bytes[0] > CACHE_GB * 1e9 and len(_audio_cache) > 1: _cache_bytes[0] -= _audio_cache.popitem(last=False)[1].nbytes
    y = y16.astype(np.float32); L = int(MAX_AUDIO_S * SR)
    if len(y) > L:
        s = np.random.randint(0, len(y) - L + 1) if train else 0; y = y[s:s + L]
    return y

tok = AutoTokenizer.from_pretrained(TEXT_CKPT); _tok_lock = threading.Lock()
def collate(idx, train):
    ys = list(_pool.map(lambda i: get_audio(i, train), idx)); lens = torch.tensor([len(y) for y in ys]); T = int(lens.max())
    wav = torch.zeros(len(ys), T)
    for k, y in enumerate(ys):
        z = (y - y.mean()) / (y.std() + 1e-7); wav[k, :len(y)] = torch.from_numpy(z)     # waveform normalised for the encoder only
    sc = torch.tensor(np.stack(list(_pool.map(scalar_feats, ys))))                              # raw intensity kept as scalars
    with _tok_lock: tk = tok([df.text.iat[i] or "" for i in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAX_TOK)   # fast tokenizers are not thread-safe
    return wav, lens, sc, tk, torch.tensor(Y[idx]), torch.tensor([float(df.path.iat[i] != "") for i in idx])

def load_fp32(name):
    # transformers >= 5 loads checkpoints in their stored dtype (often fp16); force fp32 weights (autocast handles bf16 on CUDA)
    try: return AutoModel.from_pretrained(name, dtype=torch.float32)
    except TypeError: return AutoModel.from_pretrained(name, torch_dtype=torch.float32)

class SegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.audio = load_fp32(AUDIO_CKPT); self.text = load_fp32(TEXT_CKPT)
        if hasattr(self.audio.config, 'layerdrop'): self.audio.config.layerdrop = 0.0       # random layer skipping would shorten hidden_states and break the layer mixer
        for p in list(self.audio.parameters()) + list(self.text.parameters()): p.requires_grad = False
        layers = self.audio.encoder.layers
        for l in layers[len(layers) - UNFREEZE_AUDIO_TOP:]:
            for p in l.parameters(): p.requires_grad = True
        if USE_LORA_TEXT:
            from peft import LoraConfig, get_peft_model
            self.text = get_peft_model(self.text, LoraConfig(r=8, lora_alpha=16, target_modules=["query_proj", "value_proj"], lora_dropout=0.1))
        else:
            tl = self.text.encoder.layer
            for l in tl[len(tl) - UNFREEZE_TEXT_TOP:]:
                for p in l.parameters(): p.requires_grad = True
        Ha, Ht = self.audio.config.hidden_size, self.text.config.hidden_size
        self.lw = nn.Parameter(torch.zeros(self.audio.config.num_hidden_layers + 1)); self.att = nn.Linear(Ha, 1)
        self.a = nn.Sequential(nn.LayerNorm(Ha), nn.Linear(Ha, D), nn.GELU(), nn.Dropout(0.2))
        self.t = nn.Sequential(nn.LayerNorm(Ht), nn.Linear(Ht, D), nn.GELU(), nn.Dropout(0.2))
        self.s = nn.Sequential(nn.LayerNorm(N_SCALAR), nn.Linear(N_SCALAR, 32), nn.GELU())
        self.fuse = nn.Sequential(nn.Linear(2 * D + 32 + 2, D), nn.GELU(), nn.Dropout(0.3))
        self.heads = nn.ModuleDict({t: nn.Linear(D, len(c)) for t, c in TASKS.items()})

    def encode_audio(self, wav, lens):
        hs = torch.stack(self.audio(wav, output_hidden_states=True).hidden_states)           # (L, B, T', H)
        h = (hs * torch.softmax(self.lw, 0)[:, None, None, None]).sum(0)
        fl = self.audio._get_feat_extract_output_lengths(lens.to(wav.device)).clamp(max=h.shape[1])
        mask = torch.arange(h.shape[1], device=h.device)[None] < fl[:, None]
        w = self.att(h).squeeze(-1).masked_fill(~mask, -1e4).softmax(1)                       # attentive pooling over frames
        return (h * w.unsqueeze(-1)).sum(1)

    def encode_text(self, tk):
        h = self.text(**tk).last_hidden_state; m = tk["attention_mask"].unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1)

    def forward(self, wav, lens, sc, tk, ma, mt):
        za = self.a(self.encode_audio(wav, lens)) * ma[:, None]; zt = self.t(self.encode_text(tk)) * mt[:, None]
        z = self.fuse(torch.cat([za, zt, self.s(sc) * ma[:, None], ma[:, None], mt[:, None]], 1))        # scalars are acoustic: off with the audio branch
        return z, {t: h(z) for t, h in self.heads.items()}

def masks(B, train, mode="fused", ha=None):
    ma, mt = torch.ones(B, device=DEVICE), torch.ones(B, device=DEVICE)
    if ha is not None: ma = ma * ha.to(DEVICE)                                               # rows without audio
    if mode == "audio": mt = mt * 0
    if mode == "text": ma = ma * 0
    if train:
        r = torch.rand(B, device=DEVICE); ma = ma * (r >= P_MOD_DROP).float(); mt = mt * ((r < P_MOD_DROP) | (r >= 2 * P_MOD_DROP)).float()
    return ma, mt

def prefetch(batches, train, depth=3):
    """collate upcoming batches on background threads while the GPU works on the current one"""
    ex = ThreadPoolExecutor(depth); q = deque()
    try:
        for b in batches:
            q.append(ex.submit(collate, b, train))
            if len(q) > depth: yield q.popleft().result()
        while q: yield q.popleft().result()
    finally:
        for f in q: f.cancel()
        ex.shutdown(wait=False)

def to_dev(batch):
    wav, lens, sc, tk, y, ha = batch
    nb = DEVICE == "cuda"
    return wav.to(DEVICE, non_blocking=nb), lens, sc.to(DEVICE, non_blocking=nb), {k: v.to(DEVICE, non_blocking=nb) for k, v in tk.items()}, y.to(DEVICE, non_blocking=nb), ha.to(DEVICE, non_blocking=nb)

def masked_loss(logits, y, weights):
    tot, n = 0.0, 0
    for j, t in enumerate(TASKS):
        yy = y[:, j]
        if t in STAGE_A_TASKS and (yy >= 0).any():
            tot = tot + F.cross_entropy(logits[t], yy, ignore_index=-1, weight=weights.get(t), label_smoothing=0.05); n += 1
    return tot / max(n, 1) if n else None

In [ ]:
seed_all(SEED)
model = SegNet().to(DEVICE)
tr_idx = np.where(df.split == "train")[0]; va_idx = np.where(df.split == "val")[0]; te_idx = np.where(df.split == "test")[0]
cw = {}
for j, t in enumerate(TASKS):
    yy = Y[tr_idx, j]; yy = yy[yy >= 0]
    if len(yy): c = np.bincount(yy, minlength=len(TASKS[t])).clip(1); w = 1.0 / c; cw[t] = torch.tensor(w / w.mean(), dtype=torch.float32, device=DEVICE)
enc_params = [p for n, p in model.named_parameters() if p.requires_grad and (n.startswith("audio.") or n.startswith("text."))]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and not (n.startswith("audio.") or n.startswith("text."))]
print(f"trainable: encoders {sum(p.numel() for p in enc_params) / 1e6:.1f}M, heads/fusion {sum(p.numel() for p in head_params) / 1e6:.2f}M")
opt = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENC}, {"params": head_params, "lr": LR_HEAD}], weight_decay=1e-2, fused=DEVICE == "cuda")
# Equal-weight sampling: each epoch draws PER_DS_PER_EPOCH rows from EVERY dataset. A dataset larger than that is walked through a shuffled
# permutation that continues across epochs (no row is skipped; it is seen once before any repeats); a smaller one wraps around (recycled).
pools = {d: tr_idx[df.dataset.values[tr_idx] == d] for d in sorted(set(df.dataset.values[tr_idx]))}
_perm = {d: [np.random.permutation(len(p)), 0] for d, p in pools.items()}; seen = {d: set() for d in pools}
def draw_epoch():
    out = []
    for d, p in pools.items():
        take = []
        while len(take) < PER_DS_PER_EPOCH:
            perm, k = _perm[d]
            if k >= len(perm): _perm[d] = [np.random.permutation(len(p)), 0]; continue
            n = min(PER_DS_PER_EPOCH - len(take), len(perm) - k); take += list(perm[k:k + n]); _perm[d][1] = k + n
        seen[d].update(take); out += list(p[take])
    return np.random.permutation(out)
print({d: f"{len(p)} train rows" for d, p in pools.items()}, "| draws per dataset per epoch:", PER_DS_PER_EPOCH)
autocast = torch.autocast(device_type="cuda", dtype=torch.bfloat16) if DEVICE == "cuda" else torch.autocast(device_type="cpu", enabled=False)

@torch.no_grad()
def run_eval(idx, mode="fused", return_z=False):
    model.eval(); P = {t: [] for t in TASKS}; Z = []
    for b in prefetch([idx[i:i + EVAL_BATCH] for i in range(0, len(idx), EVAL_BATCH)], False):
        wav, lens, sc, tk, y, ha = to_dev(b); ma, mt = masks(len(wav), False, mode, ha)
        with autocast: z, lg = model(wav, lens, sc, tk, ma, mt)
        for t in TASKS: P[t].append(lg[t].float().softmax(-1).cpu().numpy())
        Z.append(z.float().cpu().numpy())
    P = {t: np.concatenate(v) for t, v in P.items()}
    return (P, np.concatenate(Z)) if return_z else P

def val_score(P, idx):
    s = []
    for j, t in enumerate(TASKS):
        if t not in STAGE_A_TASKS: continue
        m = Y[idx, j] >= 0
        if m.any(): s.append((P[t].argmax(1)[m] == Y[idx, j][m]).mean())
    return float(np.mean(s)) if s else 0.0

if len(va_idx) > VAL_MAX: va_idx = np.sort(np.random.default_rng(SEED).choice(va_idx, VAL_MAX, replace=False))

# ---- Time budget: measure real speed, then size Stage A so the whole notebook ends by T_START + TIME_BUDGET_MIN ----------------
budget = None; stage_a_end = float("inf")
if TIME_BUDGET_MIN:
    def _train_steps(ix):
        model.train()
        for b in prefetch([ix[i:i + BATCH] for i in range(0, len(ix), BATCH)], True):
            wav, lens, sc, tk, y, ha = to_dev(b); ma, mt = masks(len(wav), True, ha=ha)
            with autocast: z, lg = model(wav, lens, sc, tk, ma, mt)
            loss = masked_loss({k: v.float() for k, v in lg.items()}, y, cw)
            if loss is not None: opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if DEVICE == "cuda": torch.cuda.synchronize()
    _rs = np.random.default_rng(SEED + 1)
    for g in opt.param_groups: g["lr"] *= 0.1                                              # speed calibration runs at 10% LR (no warm-up yet)
    _train_steps(_rs.choice(tr_idx, 5 * BATCH))                                         # warm-up (CUDA init); these are real training steps
    t = time.time(); _train_steps(_rs.choice(tr_idx, 25 * BATCH)); s_batch = (time.time() - t) / 25
    t = time.time(); run_eval(_rs.choice(np.arange(len(df)), 256, replace=False)); s_eval = (time.time() - t) / 256
    t = time.time(); list(_pool.map(lambda i: scalar_feats(get_audio(int(i), False)), _rs.choice(np.arange(len(df)), 400, replace=False))); s_load = (time.time() - t) / 400
    for g in opt.param_groups: g["lr"] *= 10
    t = time.time(); list(_pool.map(turn_signals, _rs.choice(np.arange(len(df)), 400, replace=False))); s_sig = (time.time() - t) / 400
    n_ds = len(pools); n_te = len(te_idx)
    def post_cost(audit_n):   # Stage B embeddings of every row + 3 test passes + shortcut audit + Stage B / report / save
        return 1.15 * (s_eval * (len(df) + 3 * n_te) + s_sig * len(df) + s_load * (audit_n + n_te)) + 300
    left = T_START + 60 * TIME_BUDGET_MIN - time.time()
    while AUDIT_MAX > 3000 and left - post_cost(AUDIT_MAX) < 0.4 * left: AUDIT_MAX = max(3000, AUDIT_MAX // 2)   # keep >= 40% of what is left for training
    stage_a_s = left - post_cost(AUDIT_MAX); val_s = s_eval * len(va_idx)
    full_s = EPOCHS_A * (n_ds * PER_DS_PER_EPOCH / BATCH * s_batch + val_s)
    if stage_a_s < full_s:
        EPOCHS_A = 4 if stage_a_s > 4 * (val_s + 60) else 2
        PER_DS_PER_EPOCH = int(max(BATCH, (stage_a_s / EPOCHS_A - val_s) / s_batch * BATCH / n_ds))
    stage_a_end = time.time() + max(stage_a_s, 0)
    budget = {"limit_min": TIME_BUDGET_MIN, "s_per_train_batch": round(s_batch, 3), "s_per_eval_row": round(s_eval, 4), "s_per_audio_load": round(s_load, 4), "s_per_turn_signals": round(s_sig, 4),
              "min_left_at_stage_a": round(left / 60, 1), "stage_a_min": round(stage_a_s / 60, 1), "reserved_after_stage_a_min": round(post_cost(AUDIT_MAX) / 60, 1),
              "per_dataset_per_epoch": PER_DS_PER_EPOCH, "epochs_a": EPOCHS_A, "audit_max": AUDIT_MAX, "val_rows": int(len(va_idx))}
    print("TIME BUDGET:", json.dumps(budget))
    if stage_a_s < 600: print("WARNING: under 10 min left for Stage A; the notebook will overrun", TIME_BUDGET_MIN, "min or train very little")

import math
n_steps = EPOCHS_A * math.ceil(len(pools) * PER_DS_PER_EPOCH / BATCH); n_warm = max(1, int(0.05 * n_steps))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / n_warm) * (0.1 + 0.45 * (1 + math.cos(math.pi * min(s, n_steps) / n_steps))))
print(f"Stage A plan: {EPOCHS_A} epochs x {len(pools)} datasets x {PER_DS_PER_EPOCH} rows, batch {BATCH} = {n_steps} steps (warm-up {n_warm}, cosine to 10% LR)")
best, best_state = -1, None
for ep in range(EPOCHS_A):
    if time.time() > stage_a_end: print("time budget reached: stopping Stage A before epoch", ep + 1); break
    model.train(); order = draw_epoch(); t0 = time.time(); tl = []; nb_ = (len(order) + BATCH - 1) // BATCH
    for bi, b in enumerate(prefetch([order[i:i + BATCH] for i in range(0, len(order), BATCH)], True)):
        if time.time() > stage_a_end + 60: print("time budget reached: epoch cut short"); break
        if bi % 200 == 0 and bi: el = time.time() - t0; print(f"  epoch {ep + 1}: batch {bi}/{nb_}, {el / bi:.2f}s/batch, epoch ETA {el / bi * (nb_ - bi) / 60:.1f} min", flush=True)
        wav, lens, sc, tk, y, ha = to_dev(b); ma, mt = masks(len(wav), True, ha=ha)
        with autocast: z, lg = model(wav, lens, sc, tk, ma, mt)
        loss = masked_loss({t: v.float() for t, v in lg.items()}, y, cw)
        if loss is None: continue
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); tl.append(float(loss))
    sv = val_score(run_eval(va_idx), va_idx) if len(va_idx) else 0.0
    print(f"epoch {ep + 1}/{EPOCHS_A} loss {np.mean(tl):.3f} val {sv:.3f} ({time.time() - t0:.0f}s)")
    if sv >= best: best, best_state = sv, {k: v.detach().clone() for k, v in model.state_dict().items() if k in dict(model.named_parameters()) and dict(model.named_parameters())[k].requires_grad}
if best_state:
    sd = model.state_dict(); sd.update(best_state); model.load_state_dict(sd)
coverage = {d: round(len(seen[d]) / len(pools[d]), 3) for d in pools}; print("fraction of each dataset's train rows used at least once:", coverage)


## 3. Stage B: speaker history + conversation events + causal temporal model
For each speaker in a call, in time order: the segment embedding `z`, the **delta from that speaker's own running mean of earlier turns** (call history), and conversation events (log words, is-question, turn index, response latency to the previous other-speaker turn where times exist). A **causal GRU** (no future context, so it can run live) produces a trajectory state per turn; the same masked multi-target heads read it. Compared against the segment-only heads on the same held-out groups.

In [ ]:
all_idx = np.arange(len(df)); Pa, Zall = run_eval(all_idx, "fused", return_z=True)
t = time.time(); SIG_raw = np.stack(list(_pool.map(turn_signals, all_idx))); print(f"measured turn signals for {len(df)} rows in {time.time() - t:.0f}s")
_trm = df.split.values == "train"; _mu, _sd = SIG_raw[_trm].mean(0), SIG_raw[_trm].std(0) + 1e-6; _sd[-1], _mu[-1] = 1.0, 0.0
SIGN = (SIG_raw - _mu) / _sd                                                                       # normalised measured signals
S_sig = np.concatenate([np.concatenate([Pa[t] for t in STAGE_A_TASKS], 1), SIGN], 1).astype(np.float32)   # Stage A signal probabilities + measured signals
SIG_delta = np.zeros_like(SIG_raw)                                                                 # raw delta vs the speaker's own earlier turns (reported)
def events(i, prev_end):
    txt = df.text.iat[i] or ""; ts, te = df.t_start.iat[i], df.t_end.iat[i]
    lat = float(np.clip(ts - prev_end, 0, 20)) if (ts == ts and prev_end is not None) else 0.0
    return [np.log1p(len(txt.split())), float("?" in txt), 0.0, lat, float(ts == ts)]

seqs = []       # one per (call, speaker): ordered segment indices + features
_calls = {c: g.sort_values("order") for c, g in df.groupby("call_id")}
for (cid, spk), sub in df.sort_values("order").groupby(["call_id", "speaker"], sort=False):
    ix = sub.index.values; callsub = _calls[cid]
    feats, run, run_s, run_r = [], np.zeros(Zall.shape[1]), np.zeros(S_sig.shape[1]), np.zeros(SIG_raw.shape[1])
    for k, i in enumerate(ix):
        other = callsub[(callsub.speaker != spk) & (callsub.order < df.order.iat[i])]
        prev_end = other.t_end.iloc[-1] if len(other) and other.t_end.iloc[-1] == other.t_end.iloc[-1] else None
        delta = Zall[i] - (run / k if k else Zall[i]); run += Zall[i]
        dS = S_sig[i] - (run_s / k if k else S_sig[i]); run_s += S_sig[i]                           # tone drift, rate change, hesitation change, emotion shift ...
        SIG_delta[i] = SIG_raw[i] - (run_r / k if k else SIG_raw[i]); run_r += SIG_raw[i]
        e = events(i, prev_end); e[2] = min(k / 50.0, 1.0)
        feats.append(np.concatenate([Zall[i], delta, S_sig[i], dS, e]))
    seqs.append((ix, np.array(feats, dtype=np.float32)))

class Temporal(nn.Module):
    def __init__(self, din):
        super().__init__(); self.inp = nn.Sequential(nn.LayerNorm(din), nn.Linear(din, D), nn.GELU()); self.gru = nn.GRU(D, D, batch_first=True)
        self.heads = nn.ModuleDict({t: nn.Linear(2 * D, len(c)) for t, c in TASKS.items()})
    def forward(self, x):
        h0 = self.inp(x); g, _ = self.gru(h0); g = torch.cat([g, h0], -1); return {t: h(g) for t, h in self.heads.items()}

def batch_seq(items):
    L = max(len(f) for _, f in items); X = torch.zeros(len(items), L, items[0][1].shape[1]); Yb = torch.full((len(items), L, len(TASKS)), -1, dtype=torch.long)
    for b, (ix, f) in enumerate(items): X[b, :len(f)] = torch.from_numpy(f); Yb[b, :len(f)] = torch.from_numpy(Y[ix])
    return X.to(DEVICE), Yb.to(DEVICE)

split_of = lambda ix: df.split.iat[ix[0]]
tr_s = [s for s in seqs if split_of(s[0]) == "train"]; va_s = [s for s in seqs if split_of(s[0]) == "val"]; te_s = [s for s in seqs if split_of(s[0]) == "test"]
seed_all(SEED); tmp = Temporal(seqs[0][1].shape[1]).to(DEVICE); opt2 = torch.optim.AdamW(tmp.parameters(), lr=1e-3, weight_decay=1e-2)
def temporal_loss(lg, Yb):
    tot, n = 0.0, 0
    for j, t in enumerate(TASKS):
        yy = Yb[..., j].reshape(-1)
        if (yy >= 0).any(): tot = tot + F.cross_entropy(lg[t].reshape(-1, len(TASKS[t])), yy, ignore_index=-1, weight=cw.get(t)); n += 1
    return tot / n if n else None
for ep in range(EPOCHS_B):
    tmp.train(); random.shuffle(tr_s)
    for i in range(0, len(tr_s), 16):
        X, Yb = batch_seq(tr_s[i:i + 16]); loss = temporal_loss(tmp(X), Yb)
        if loss is not None: opt2.zero_grad(); loss.backward(); opt2.step()
print("temporal model trained on", len(tr_s), "speaker sequences")
@torch.no_grad()
def temporal_predict(items):
    tmp.eval(); P = {t: np.zeros((len(df), len(TASKS[t]))) for t in TASKS}
    for ix, f in items:
        lg = tmp(torch.from_numpy(f)[None].to(DEVICE))
        for t in TASKS: P[t][ix] = lg[t][0].softmax(-1).cpu().numpy()
    return P

## 4. Evaluation (held-out groups only)
For every target that has labels in the test groups: accuracy and macro-F1 with bootstrap CIs over groups, for **text-only, audio-only, fused, fused + temporal**. Binary targets add precision, recall, false-positive and false-negative rate, calibration (ECE) and **abstention** (answer only when max probability >= 0.7: coverage and accuracy on answered). Audio ablations mask the audio branch only; the transcript is untouched.

In [ ]:
from sklearn.metrics import f1_score, precision_recall_fscore_support
def boot(vals_by_group, n=1000):
    v = np.array(list(vals_by_group.values())); r = np.random.default_rng(SEED)
    return [round(float(np.percentile([r.choice(v, len(v)).mean() for _ in range(n)], q)), 3) for q in (2.5, 97.5)] if len(v) > 1 else None
def ece(p, y, bins=10):
    c, pr = p.max(1), p.argmax(1); e = 0.0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (c > lo) & (c <= lo + 1 / bins)
        if m.any(): e += m.mean() * abs((pr[m] == y[m]).mean() - c[m].mean())
    return round(float(e), 3)

PRED = {"text": run_eval(te_idx, "text"), "audio": run_eval(te_idx, "audio"), "fused": run_eval(te_idx, "fused")}
tp = temporal_predict(te_s); PRED["fused+temporal"] = {t: tp[t][te_idx] for t in TASKS}
report = {}
for j, t in enumerate(TASKS):
    y = Y[te_idx, j]; m = y >= 0
    if m.sum() < 5: report[t] = {"status": "no labelled test data (head untrained or unevaluated)"}; continue
    grp = df.group.values[te_idx][m]; report[t] = {"n_test": int(m.sum()), "n_groups": int(len(set(grp))), "majority": round(float(np.bincount(y[m]).max() / m.sum()), 3)}
    for mode, P in PRED.items():
        if t not in STAGE_A_TASKS and mode != "fused+temporal": continue                     # Stage A does not train deception heads
        pr = P[t][m].argmax(1); acc = {g: float((pr[grp == g] == y[m][grp == g]).mean()) for g in set(grp)}
        r = {"acc": round(float((pr == y[m]).mean()), 3), "macro_f1": round(float(f1_score(y[m], pr, average="macro")), 3), "ci95_over_groups": boot(acc), "ece": ece(P[t][m], y[m])}
        conf = P[t][m].max(1); keep = conf >= 0.7; r["abstain_coverage"] = round(float(keep.mean()), 3); r["acc_when_answering"] = round(float((pr[keep] == y[m][keep]).mean()), 3) if keep.any() else None
        if len(TASKS[t]) == 2 and len(set(y[m])) == 2:
            pp, rr, _, _ = precision_recall_fscore_support(y[m], pr, average="binary", zero_division=0)
            r.update(precision=round(float(pp), 3), recall=round(float(rr), 3), fpr=round(float(((pr == 1) & (y[m] == 0)).sum() / max((y[m] == 0).sum(), 1)), 3), fnr=round(float(((pr == 0) & (y[m] == 1)).sum() / max((y[m] == 1).sum(), 1)), 3))
        report[t][mode] = r
pd.json_normalize(report, sep=".").T

In [ ]:
# ---- Shortcut audit (brief: avoid leakage from recording setup, role, script) -------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
_rs = np.random.default_rng(SEED); _tr = np.where(df.split.values == "train")[0]
audit_idx = np.concatenate([_rs.choice(_tr, min(len(_tr), AUDIT_MAX), replace=False), np.where(df.split.values == "test")[0]])   # scalar-only classifier: <=30k train rows (documented in the results JSON)
S_all = np.zeros((len(df), N_SCALAR), np.float32)
for i, s in zip(audit_idx, _pool.map(lambda i: scalar_feats(get_audio(i, False)), audit_idx)): S_all[i] = s
audit = {}
for j, t in enumerate(TASKS):
    trm = (Y[:, j] >= 0) & (df.split.values == "train") & np.isin(np.arange(len(df)), audit_idx); tem = (Y[:, j] >= 0) & (df.split.values == "test")
    if trm.sum() > 20 and tem.sum() > 5 and len(set(Y[trm, j])) > 1:
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(S_all[trm], Y[trm, j])
        audit[t] = {"scalar_channel_features_only_acc": round(float((clf.predict(S_all[tem]) == Y[tem, j]).mean()), 3),
                    "model_acc": report.get(t, {}).get("fused" if t in STAGE_A_TASKS else "fused+temporal", {}).get("acc")}
print("If scalar-only is close to the model, the label is predictable from loudness/channel alone (shortcut):"); print(json.dumps(audit, indent=1))

## 5. Output contract (brief), live hints, saved artefacts
`assess_call(call_id)` returns the brief's JSON. Targets whose head saw fewer than `N_MIN_LABELS` labels, or had no labelled test data, return `null` scores and `status: insufficient_evidence`. `calibrated_probability` is always `null` and `calibration_status` is `not_established`: these are model scores, not percentages of actual deception.

In [ ]:
Pall = temporal_predict(seqs)
n_lab = {t: int((Y[tr_idx, j] >= 0).sum()) for j, t in enumerate(TASKS)}
trained = {t: n_lab[t] >= N_MIN_LABELS and "fused+temporal" in report.get(t, {}) for t in TASKS}
ALT = ["rehearsal or reading a prepared answer", "uncertainty or lack of information", "topic or channel change", "recording conditions (distance, gain, noise)", "emotion unrelated to the call"]

def live_hint(cid, speaker):
    sub = df[(df.call_id == cid) & (df.speaker == speaker)].sort_values("order")
    if len(sub) < 4: return None
    S = np.stack([scalar_feats(get_audio(i, False)) for i in sub.index]); base, cur = S[:-1], S[-1]
    z = (cur - base.mean(0)) / np.maximum(base.std(0), np.array([0.5, 2.0, 0.05, 0.01, 0.05, 0.01, 2.0]))
    obs = [f"{n} {'up' if v > 0 else 'down'} vs this speaker's earlier turns (z {v:+.1f})" for n, v in zip(["duration", "loudness", "peak", "clipping", "pause fraction", "zero-crossing rate", "loudness range"], z) if abs(v) >= 2.5]
    return {"speaker": speaker, "evidence_segment_id": int(sub.index[-1]), "acoustic_observations": obs, "comparison_interval": f"turns 1-{len(sub) - 1}",
            "hypothesis": "possible change in delivery; may reflect " + ", ".join(ALT[:3]) + ". Ask a concrete follow-up."} if obs else None

def assess_call(cid):
    ix = df.index[df.call_id == cid].values; out = {"call_id": cid, "assessment_scope": "commercial_conversation", "deception_assessment": {}, "delivery_assessment": {},
                                                    "emotion_timeline": [], "live_hints": [], "post_call_summary": None}
    for t, target in (("bluff", "negotiating_bluff"), ("fabrication", "fabrication"), ("mismatch", "expressed_interest_vs_verified_intent_mismatch")):
        if trained[t]:
            s = Pall[t][ix, 1]; top = ix[np.argsort(-s)[:3]]
            out["deception_assessment"][target] = {"model_score": round(float(s.max()), 3), "calibrated_probability": None, "calibration_status": "not_established",
                                                   "evidence_segment_ids": [int(i) for i in top], "alternative_explanations": ALT, "status": "model_score_only"}
        else:
            out["deception_assessment"][target] = {"model_score": None, "calibrated_probability": None, "calibration_status": "not_established",
                                                   "evidence_segment_ids": [], "alternative_explanations": [], "status": "insufficient_evidence"}
    out["delivery_assessment"] = {"read_or_rehearsed_score": round(float(1 - Pall["delivery"][ix, 2].mean()), 3) if trained["delivery"] else None, "evidence_segment_ids": []}
    if trained["emotion"]:
        out["emotion_timeline"] = [{"segment_id": int(i), "speaker": df.speaker.iat[i], "probs": {c: round(float(p), 3) for c, p in zip(EMO, Pall["emotion"][i])}} for i in ix]
    out["signal_timeline"] = [{"segment_id": int(i), "speaker": df.speaker.iat[i], "t_start": None if df.t_start.iat[i] != df.t_start.iat[i] else float(df.t_start.iat[i]),
                               "signals": {n: round(float(v), 3) for n, v in zip(SIG_NAMES, SIG_raw[i])},
                               "delta_vs_own_earlier_turns": {n: round(float(v), 3) for n, v in zip(SIG_NAMES, SIG_delta[i])}} for i in ix]
    out["live_hints"] = [h for h in (live_hint(cid, s) for s in df.speaker[ix].unique()) if h]
    return out

demo_call = df[df.split == "test"].groupby("call_id").size().idxmax()
example = assess_call(demo_call); print(json.dumps({k: example[k] for k in ("call_id", "deception_assessment", "delivery_assessment")}, indent=1)[:1800])

In [ ]:
tag = "quick" if QUICK else "full"
torch.save({"trainable": {k: v.cpu() for k, v in model.state_dict().items() if k in dict(model.named_parameters()) and dict(model.named_parameters())[k].requires_grad},
            "temporal": tmp.state_dict(), "tasks": TASKS, "audio_ckpt": AUDIO_CKPT, "text_ckpt": TEXT_CKPT, "unfreeze_audio_top": UNFREEZE_AUDIO_TOP,
            "unfreeze_text_top": UNFREEZE_TEXT_TOP, "seed": SEED, "quick": QUICK}, MODELS / f"baseline_brief_{tag}.pt")
res = {"label": "EXPLORATORY", "quick": QUICK, "full_data": FULL_DATA, "ami_shards": AMI_SHARDS or f"{AMI_MAX_SHARDS} evenly spaced of 51", "librispeech_sets": LIBRI_SETS, "per_dataset_per_epoch": PER_DS_PER_EPOCH, "epochs_a": EPOCHS_A, "batch": BATCH, "lr_enc_head": [LR_ENC, LR_HEAD], "lr_schedule": "5% warm-up, cosine to 10%", "stage_a_tasks": list(STAGE_A_TASKS), "stage_b_inputs": "Stage A signal probs + measured signals " + str(SIG_NAMES) + " + deltas vs own earlier turns + fused speech embedding + its delta + conversation events", "time_budget": budget, "audit_train_rows": int(min(len(np.where(df.split.values == "train")[0]), AUDIT_MAX)), "val_rows_for_epoch_selection": int(len(va_idx)), "total_wall_min": round((time.time() - T_START) / 60, 1), "train_row_coverage": coverage, "device": DEVICE, "datasets": DATASETS, "n_segments": len(df), "n_calls": int(df.call_id.nunique()), "labels_in_train": n_lab,
       "targets_with_trained_head": trained, "unfrozen": {"audio_top_layers": UNFREEZE_AUDIO_TOP, "text_top_layers": UNFREEZE_TEXT_TOP, "lora_text": USE_LORA_TEXT},
       "report": report, "shortcut_audit": audit, "example_output": example}
(OUT / f"brief_architecture_{tag}.json").write_text(json.dumps(res, indent=1, default=str)); print("saved", MODELS / f"baseline_brief_{tag}.pt", OUT / f"brief_architecture_{tag}.json")

## 6. What to do next with the GPU
1. `QUICK = False`, run all cells (MELD + AMI 10 shards + LibriSpeech dev/test-clean + RAVDESS + Diplomacy + synthetic). Download data and models untimed first; `TIME_BUDGET_MIN = 90` then sizes Stage A from measured speed (`docs/train_on_gpu_machine.md`). First full run on the 5090: in progress (0.145 s/batch, full 12 x 8000 schedule fits), results pending.
2. Add a verified-ground-truth deception adapter in `load_deception()` (see `docs/research/deception_delivery_datasets.md`); until then Stage B's optional `bluff/fabrication/mismatch` readings rest on text-only (Diplomacy) or TTS (ElevenLabs) labels; bluff (6 labels) and mismatch (12) stay `null` until more negotiation calls are generated.
3. Compare Stage B against an LLM reading `signal_timeline` (same held-out calls); keep whichever wins on held-out groups.
4. Try `AUDIO_CKPT = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"` (research-only licence) and `UNFREEZE_AUDIO_TOP` 4 / 6 / 8; keep the one that wins on held-out groups, not on training loss.
5. If the fine-tuned model does not beat the frozen-probe baseline (`results/probe_wavlm_ravdess.json`, 0.751 acted 8-class), ship the simpler one and say so.
6. Mic/noise/gain robustness sweeps and false-alarm rates on calm and anxious speech are still to do (brief, evaluation section).